# Yassine AI — Phase 1: A Neural Network From Scratch (NumPy only)

**Goal:** build, train and evaluate a handwritten-digit classifier where *you* wrote every piece of the learning process: forward pass, loss, backpropagation and gradient descent.

| What this notebook is | What it is NOT |
|---|---|
| ✅ A neural network **implemented from scratch** (NumPy only, no PyTorch/TensorFlow) | ❌ Fine-tuning a pretrained model |
| ✅ A model **trained from scratch** (random initial weights, learns only from MNIST) | ❌ Calling an external AI API (OpenAI, Claude, …) |

Libraries used: **NumPy** (all the math of the model), **Matplotlib** (plots), **scikit-learn** (only for the train/validation split and evaluation reports).

---

## The architecture (Yassine AI v1)

```
 input image 28×28            hidden layer              output layer
 = 784 pixel values  ──►  128 neurons + ReLU  ──►  10 neurons + softmax  ──►  P(digit = 0..9)
        X                  Z1 = X·W1 + b1            Z2 = A1·W2 + b2
                           A1 = ReLU(Z1)             P  = softmax(Z2)
```

- **Parameters:** `W1` (784×128), `b1` (128), `W2` (128×10), `b2` (10) → **101,770 numbers** to learn.
- **Learning:** we measure how wrong the predictions are (**cross-entropy loss**), compute how each parameter influenced that error (**backpropagation**, i.e. the chain rule), and nudge each parameter in the direction that reduces the error (**gradient descent**). Repeat ~15,000 times.

## How to run this notebook
1. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this `.ipynb` file.
2. A free CPU runtime is enough (no GPU needed — NumPy doesn't use the GPU anyway).
3. Run the cells **in order, top to bottom** with **Shift + Enter**. Read the text cell before each code cell.
4. Total runtime: roughly 2–4 minutes on free Colab.

If something breaks: **Runtime → Restart session**, then run everything again from the top (**Runtime → Run all**).

## Cell 1 — Setup and reproducibility

A neural network starts from **random** weights and sees the data in a **random** order. If we fix the *random seed*, every run produces exactly the same numbers, which makes debugging possible ("did my change help, or was it luck?").

**Expected output:** the NumPy version and `Setup complete`.

In [ ]:
import os
import json
import time
import urllib.request

import numpy as np
import matplotlib.pyplot as plt

SEED = 42  # change it to see how much results vary between random initializations
np.set_printoptions(precision=4, suppress=True)

print("NumPy version:", np.__version__)
print("Setup complete")

## Cell 2 — Load the MNIST dataset

**MNIST** = 70,000 grayscale images (28×28 pixels) of handwritten digits 0–9, each with the correct label. It is the classic "hello world" of machine learning and is freely usable for research and education.

- 60,000 images are the official **training** set.
- 10,000 images are the official **test** set, written by *different people*. The model will never learn from them.

We download the raw arrays (an ~11 MB `.npz` file). If that download fails, the code falls back to scikit-learn's OpenML loader (slower, ~1–2 minutes).

**Expected output:** `x_train_full: (60000, 28, 28) uint8`, `x_test: (10000, 28, 28) uint8`.

In [ ]:
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
MNIST_PATH = "mnist.npz"


def load_mnist(path=MNIST_PATH):
    # Returns (x_train, y_train, x_test, y_test) as uint8 arrays.
    try:
        if not os.path.exists(path):
            print("Downloading MNIST (~11 MB)...")
            urllib.request.urlretrieve(MNIST_URL, path)
        with np.load(path) as data:
            return data["x_train"], data["y_train"], data["x_test"], data["y_test"]
    except Exception as error:
        print("Direct download failed:", error)
        print("Falling back to scikit-learn / OpenML (can take 1-2 minutes)...")
        from sklearn.datasets import fetch_openml
        X, y = fetch_openml("mnist_784", version=1, as_frame=False,
                            parser="liac-arff", return_X_y=True)
        X = X.reshape(-1, 28, 28).astype(np.uint8)
        y = y.astype(np.uint8)
        # OpenML keeps the original order: first 60,000 = train, last 10,000 = test
        return X[:60000], y[:60000], X[60000:], y[60000:]


x_train_full, y_train_full, x_test, y_test = load_mnist()
print("x_train_full:", x_train_full.shape, x_train_full.dtype)
print("y_train_full:", y_train_full.shape, y_train_full.dtype)
print("x_test:      ", x_test.shape, x_test.dtype)
print("y_test:      ", y_test.shape, y_test.dtype)
print("Pixel range: ", x_train_full.min(), "to", x_train_full.max())

## Cell 3 — Look at the data

Always look at your data before training. Each image is a 28×28 grid of numbers from 0 (black) to 255 (white).

**Expected output:** a row of 10 handwritten digits with their labels, and a bar chart showing that each digit appears roughly 6,000 times in the training set (the classes are balanced, so "accuracy" is a fair metric).

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train_full[i], cmap="gray")
    ax.set_title(f"label: {y_train_full[i]}")
    ax.axis("off")
plt.suptitle("First 10 training images")
plt.show()

counts = np.bincount(y_train_full, minlength=10)
plt.figure(figsize=(6, 3))
plt.bar(range(10), counts)
plt.xticks(range(10))
plt.xlabel("digit")
plt.ylabel("number of images")
plt.title("Training images per digit")
plt.show()
print("Images per digit:", counts)

## Cell 4 — Prepare the data: flatten, normalize, split, one-hot

1. **Flatten:** our simple network takes a vector, so each 28×28 image becomes a row of 784 numbers.
2. **Normalize:** divide by 255 so pixels are in `[0, 1]`. Large input values make gradients large and training unstable.
3. **Split** the 60,000 training images into:
   - **train (50,000):** the model learns from these.
   - **validation (10,000):** used *during development* to check progress and choose settings (learning rate, hidden size, epochs).
   - The **test set (10,000)** stays locked away until the very end. If we tuned settings on the test set, our final score would be optimistic — we would be "studying the exam questions".
4. **One-hot encode** the labels: the digit `3` becomes `[0,0,0,1,0,0,0,0,0,0]`, which matches the 10 output neurons.

**Expected output:** `X_train (50000, 784)`, `X_val (10000, 784)`, `X_test (10000, 784)` and the one-hot example.

In [ ]:
from sklearn.model_selection import train_test_split


def preprocess(images):
    # (N, 28, 28) uint8 in [0, 255]  ->  (N, 784) float32 in [0, 1]
    return images.reshape(len(images), -1).astype(np.float32) / 255.0


def one_hot(labels, num_classes=10):
    encoded = np.zeros((len(labels), num_classes), dtype=np.float32)
    encoded[np.arange(len(labels)), labels] = 1.0
    return encoded


X_all = preprocess(x_train_full)
X_test = preprocess(x_test)

# stratify keeps the same proportion of each digit in train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_train_full, test_size=10000, random_state=SEED, stratify=y_train_full
)

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_val:  ", X_val.shape, " y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, " y_test: ", y_test.shape)
print("Pixel range after normalization:", X_train.min(), "to", X_train.max())
print("Label", y_train[0], "as one-hot:", one_hot(y_train[:1])[0])

## Cell 5 — Activation functions: ReLU and softmax

**Why do we need an activation?** A layer computes `X·W + b`, which is a *linear* function. Stacking linear functions gives… another linear function. Without a non-linearity, 100 layers would be no smarter than 1. The activation lets the network bend and combine features (strokes, loops, corners).

**ReLU** (Rectified Linear Unit), used in the hidden layer:
$$\text{ReLU}(z) = \max(0, z) \qquad \frac{d\,\text{ReLU}}{dz} = \begin{cases}1 & z > 0\\ 0 & z \le 0\end{cases}$$

**Softmax**, used in the output layer, turns 10 raw scores ("logits") into 10 probabilities that are positive and sum to 1:
$$\text{softmax}(z)_k = \frac{e^{z_k}}{\sum_{j} e^{z_j}}$$

Numerical trick: `e^1000` overflows to infinity. Subtracting the largest score from every score first gives the **same** result (it cancels in the fraction) but never overflows.

**Expected output:** ReLU zeroes the negatives; softmax probabilities sum to 1.0, and the "huge" logits do not produce `nan`.

In [ ]:
def relu(z):
    return np.maximum(0, z)


def relu_derivative(z):
    return (z > 0).astype(z.dtype)


def softmax(z):
    z_shifted = z - z.max(axis=1, keepdims=True)  # numerical stability
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


z = np.array([[-2.0, -0.5, 0.0, 1.5, 3.0]])
print("z:               ", z[0])
print("relu(z):         ", relu(z)[0])
print("relu'(z):        ", relu_derivative(z)[0])
p = softmax(z)
print("softmax(z):      ", p[0], " sum =", p.sum())
print("softmax(huge z): ", softmax(np.array([[1000.0, 1001.0, 1002.0]]))[0])

## Cell 6 — The loss function: cross-entropy

The **loss** is a single number that says how wrong the model is. Training = making this number small.

For one image whose true class is $y$, cross-entropy is $-\log(p_y)$: minus the log of the probability the model gave to the **correct** answer.
- Model says 0.99 for the right digit → loss = 0.01 (great)
- Model says 0.10 (a random guess among 10) → loss = 2.30
- Model says 0.001 → loss = 6.9 (confidently wrong is punished hard)

Over a batch of $m$ images with one-hot labels $Y$:
$$L = -\frac{1}{m}\sum_{i=1}^{m}\sum_{k=0}^{9} Y_{ik}\,\log P_{ik}$$

**Expected output:** the three example losses above (≈0.01, ≈2.30, ≈6.91).

In [ ]:
def cross_entropy_loss(probs, y_onehot):
    eps = 1e-12  # avoids log(0) = -infinity
    return float(-np.mean(np.sum(y_onehot * np.log(probs + eps), axis=1)))


y_true = one_hot(np.array([3]))
for p_correct in [0.99, 0.10, 0.001]:
    probs = np.full((1, 10), (1 - p_correct) / 9)
    probs[0, 3] = p_correct
    print(f"P(correct) = {p_correct:<5} -> loss = {cross_entropy_loss(probs, y_true):.3f}")

## Cell 7 — The network: initialization, forward pass, backpropagation, update

This is the heart of Yassine AI. Read it slowly.

### Weight initialization
Weights start **random** (if they were all equal, every neuron would learn the same thing). Their *scale* matters: too big → activations explode; too small → signals vanish. We use:
- **He initialization** for the ReLU layer: $W_1 \sim \mathcal{N}(0,\ 2/784)$
- **Xavier/LeCun initialization** for the softmax layer: $W_2 \sim \mathcal{N}(0,\ 1/128)$
- Biases start at 0.

### Forward pass (prediction)
$$Z_1 = XW_1 + b_1 \quad\to\quad A_1 = \text{ReLU}(Z_1) \quad\to\quad Z_2 = A_1W_2 + b_2 \quad\to\quad P = \text{softmax}(Z_2)$$
We keep the intermediate values (the **cache**) because backpropagation needs them.

### Backward pass (backpropagation = the chain rule, applied layer by layer from the output back to the input)
The goal is $\frac{\partial L}{\partial W}$ for every parameter: "if I increase this weight slightly, does the loss go up or down, and how much?"

| Step | Formula | Meaning |
|---|---|---|
| 1 | $dZ_2 = \frac{1}{m}(P - Y)$ | Softmax + cross-entropy combine into this beautifully simple error: predicted minus true |
| 2 | $dW_2 = A_1^\top dZ_2,\quad db_2 = \sum_i dZ_2$ | How each output weight contributed to the error |
| 3 | $dA_1 = dZ_2 W_2^\top$ | Send the error back to the hidden layer |
| 4 | $dZ_1 = dA_1 \odot \text{ReLU}'(Z_1)$ | Neurons that were "off" (Z₁ ≤ 0) pass no gradient |
| 5 | $dW_1 = X^\top dZ_1,\quad db_1 = \sum_i dZ_1$ | How each input weight contributed |

### Gradient descent (learning)
$$W \leftarrow W - \eta \cdot \frac{\partial L}{\partial W}$$
The gradient points "uphill" (towards more error), so we step in the opposite direction. $\eta$ (eta) is the **learning rate**: the step size.

**Expected output:** the parameter shapes, `Total parameters: 101,770`, and an initial loss close to $\ln(10) \approx 2.30$ — an untrained network is guessing randomly among 10 classes.

In [ ]:
class YassineNet:
    # A 2-layer neural network: input -> hidden (ReLU) -> output (softmax).

    def __init__(self, input_size=784, hidden_size=128, output_size=10, seed=SEED, dtype=np.float32):
        rng = np.random.default_rng(seed)
        self.params = {
            # He initialization for the ReLU layer
            "W1": (rng.standard_normal((input_size, hidden_size)) * np.sqrt(2.0 / input_size)).astype(dtype),
            "b1": np.zeros(hidden_size, dtype=dtype),
            # Xavier/LeCun initialization for the softmax layer
            "W2": (rng.standard_normal((hidden_size, output_size)) * np.sqrt(1.0 / hidden_size)).astype(dtype),
            "b2": np.zeros(output_size, dtype=dtype),
        }

    def forward(self, X):
        W1, b1, W2, b2 = self.params["W1"], self.params["b1"], self.params["W2"], self.params["b2"]
        Z1 = X @ W1 + b1      # (m, 784) @ (784, 128) -> (m, 128)
        A1 = relu(Z1)
        Z2 = A1 @ W2 + b2     # (m, 128) @ (128, 10)  -> (m, 10)
        P = softmax(Z2)
        cache = {"X": X, "Z1": Z1, "A1": A1, "P": P}
        return P, cache

    def backward(self, cache, y_onehot):
        X, Z1, A1, P = cache["X"], cache["Z1"], cache["A1"], cache["P"]
        m = X.shape[0]

        dZ2 = (P - y_onehot) / m                    # step 1
        dW2 = A1.T @ dZ2                            # step 2
        db2 = dZ2.sum(axis=0)
        dA1 = dZ2 @ self.params["W2"].T             # step 3
        dZ1 = dA1 * relu_derivative(Z1)             # step 4
        dW1 = X.T @ dZ1                             # step 5
        db1 = dZ1.sum(axis=0)

        return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

    def update(self, grads, learning_rate):
        for name in self.params:
            self.params[name] -= learning_rate * grads[name]

    def predict_proba(self, X):
        return self.forward(X)[0]

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

    def num_parameters(self):
        return sum(p.size for p in self.params.values())


demo_model = YassineNet(seed=SEED)
for name, p in demo_model.params.items():
    print(f"{name}: shape {p.shape}")
print(f"Total parameters: {demo_model.num_parameters():,}")

initial_probs = demo_model.predict_proba(X_train[:1000])
print(f"Initial loss on 1000 images: {cross_entropy_loss(initial_probs, one_hot(y_train[:1000])):.4f}"
      f"  (random guessing = ln(10) = {np.log(10):.4f})")
print(f"Initial accuracy: {np.mean(initial_probs.argmax(axis=1) == y_train[:1000]):.1%}  (random guessing = 10%)")

## Cell 8 — Gradient check: prove that backpropagation is correct

Backprop bugs are sneaky: the network may still learn *a little* with a wrong gradient, so you don't notice. Professionals verify gradients numerically using the definition of the derivative:
$$\frac{\partial L}{\partial w} \approx \frac{L(w+\varepsilon) - L(w-\varepsilon)}{2\varepsilon}$$

We do this for **every** parameter of a tiny network (5 inputs → 4 hidden → 3 outputs, in float64 precision) and compare with our analytic `backward()`. The *relative error* should be tiny (below `1e-6`).

**Expected output:** four lines with relative errors around `1e-9`–`1e-11` and `Gradient check PASSED`.
If you see `FAILED`, there is a bug in `backward()` — fix it before training.

In [ ]:
def gradient_check(eps=1e-5, tolerance=1e-6, seed=0):
    rng = np.random.default_rng(seed)
    net = YassineNet(input_size=5, hidden_size=4, output_size=3, seed=seed, dtype=np.float64)
    X = rng.standard_normal((6, 5))
    Y = one_hot(rng.integers(0, 3, size=6), num_classes=3).astype(np.float64)

    _, cache = net.forward(X)
    analytic = net.backward(cache, Y)

    worst = 0.0
    for name, param in net.params.items():
        numeric = np.zeros_like(param)
        for idx in np.ndindex(param.shape):
            original = param[idx]
            param[idx] = original + eps
            loss_plus = cross_entropy_loss(net.predict_proba(X), Y)
            param[idx] = original - eps
            loss_minus = cross_entropy_loss(net.predict_proba(X), Y)
            param[idx] = original  # restore
            numeric[idx] = (loss_plus - loss_minus) / (2 * eps)
        rel_error = np.linalg.norm(numeric - analytic[name]) / (
            np.linalg.norm(numeric) + np.linalg.norm(analytic[name]) + 1e-12)
        worst = max(worst, rel_error)
        print(f"{name}: relative error = {rel_error:.2e}")

    passed = worst < tolerance
    print("Gradient check PASSED" if passed else "Gradient check FAILED - check backward()")
    return passed


gradient_check_passed = gradient_check()

## Cell 9 — Hyperparameters

**Hyperparameters** are settings *you* choose; **parameters** (weights) are what the model learns.

| Name | Value | What it controls |
|---|---|---|
| `HIDDEN_SIZE` | 128 | Capacity: more neurons can learn more patterns (and overfit more) |
| `LEARNING_RATE` | 0.1 | Step size. Too big → loss jumps around or becomes `nan`. Too small → very slow learning |
| `BATCH_SIZE` | 64 | **Mini-batch gradient descent:** each update uses 64 images, not all 50,000. Cheaper per step and the noise helps escape bad spots |
| `EPOCHS` | 20 | One epoch = the model has seen every training image once |

50,000 images / 64 per batch ≈ 782 updates per epoch → ~15,640 updates in total.

In [ ]:
HIDDEN_SIZE = 128
LEARNING_RATE = 0.1
BATCH_SIZE = 64
EPOCHS = 20

print(f"Updates per epoch: {int(np.ceil(len(X_train) / BATCH_SIZE))}")
print(f"Total updates:     {int(np.ceil(len(X_train) / BATCH_SIZE)) * EPOCHS}")

## Cell 10 — The training loop

For each epoch:
1. **Shuffle** the training set (so batches differ every epoch and the model can't memorize the order).
2. For each mini-batch: **forward → loss → backward → update**. These four lines are what "training an AI" means.
3. At the end of the epoch, measure loss and accuracy on the **train** and **validation** sets.

**Expected output** (our reference run; your last digits may differ slightly depending on hardware and NumPy version; each epoch takes ~0.5–3 s on Colab):
```
Epoch  1/20 | train loss 0.3209 acc 90.33% | val loss 0.3313 acc 90.23% | 0.5s
Epoch  2/20 | train loss 0.1954 acc 94.32% | val loss 0.2190 acc 93.72% | 0.4s
Epoch  3/20 | train loss 0.1487 acc 95.80% | val loss 0.1712 acc 95.11% | 0.4s
...
Epoch 18/20 | train loss 0.0273 acc 99.44% | val loss 0.0816 acc 97.68% | 0.4s
Epoch 19/20 | train loss 0.0254 acc 99.50% | val loss 0.0796 acc 97.74% | 0.4s
Epoch 20/20 | train loss 0.0260 acc 99.46% | val loss 0.0813 acc 97.54% | 0.4s
```
What to look for: validation accuracy climbs fast (≈90% after epoch 1, ≈95% after epoch 3) and ends around 97.5%. Train loss keeps dropping; validation loss drops, then flattens and wiggles. Small bumps (a slightly worse epoch) are normal with mini-batch noise.

In [ ]:
def evaluate(model, X, y):
    probs = model.predict_proba(X)
    loss = cross_entropy_loss(probs, one_hot(y))
    accuracy = float(np.mean(probs.argmax(axis=1) == y))
    return loss, accuracy


def train(model, X_train, y_train, X_val, y_val, epochs, batch_size, learning_rate, seed=SEED):
    rng = np.random.default_rng(seed)
    Y_train = one_hot(y_train)
    n = len(X_train)
    history = {"batch_loss": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        order = rng.permutation(n)                       # 1. shuffle

        for start in range(0, n, batch_size):            # 2. mini-batches
            idx = order[start:start + batch_size]
            probs, cache = model.forward(X_train[idx])   #    forward
            loss = cross_entropy_loss(probs, Y_train[idx])  # loss
            grads = model.backward(cache, Y_train[idx])  #    backward
            model.update(grads, learning_rate)           #    update
            history["batch_loss"].append(loss)

        train_loss, train_acc = evaluate(model, X_train, y_train)   # 3. measure
        val_loss, val_acc = evaluate(model, X_val, y_val)
        for key, value in [("train_loss", train_loss), ("train_acc", train_acc),
                           ("val_loss", val_loss), ("val_acc", val_acc)]:
            history[key].append(value)

        if not np.isfinite(train_loss):
            print("Loss became NaN/inf: the learning rate is probably too high. Stopping.")
            break

        print(f"Epoch {epoch:2d}/{epochs} | train loss {train_loss:.4f} acc {train_acc:.2%} "
              f"| val loss {val_loss:.4f} acc {val_acc:.2%} | {time.time() - start_time:.1f}s")
    return history


model = YassineNet(hidden_size=HIDDEN_SIZE, seed=SEED)  # fresh random weights
history = train(model, X_train, y_train, X_val, y_val,
                epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE)

## Cell 11 — Visualize learning

- **Left:** the loss of every mini-batch (light) and its moving average (dark). It should fall steeply at first, then slowly.
- **Middle:** train vs. validation loss per epoch.
- **Right:** train vs. validation accuracy per epoch.

**How to read the gap between train and validation:**
- Both curves close together and improving → the model is **generalizing** (learning real digit shapes).
- Train keeps improving while validation gets **worse** → **overfitting**: the model is memorizing the training images instead of learning general rules. A small gap (train 99%+ vs. val ~97.5%) is normal for this network.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
epochs_range = range(1, len(history["train_loss"]) + 1)

batch_loss = np.array(history["batch_loss"])
window = 100
smoothed = np.convolve(batch_loss, np.ones(window) / window, mode="valid")
axes[0].plot(batch_loss, alpha=0.25, label="batch loss")
axes[0].plot(np.arange(window - 1, len(batch_loss)), smoothed, label=f"moving average ({window})")
axes[0].set_xlabel("update step")
axes[0].set_ylabel("cross-entropy loss")
axes[0].set_title("Loss during training")
axes[0].set_ylim(0, 1.0)
axes[0].legend()

axes[1].plot(epochs_range, history["train_loss"], "o-", label="train")
axes[1].plot(epochs_range, history["val_loss"], "o-", label="validation")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("loss")
axes[1].set_title("Loss per epoch")
axes[1].legend()

axes[2].plot(epochs_range, history["train_acc"], "o-", label="train")
axes[2].plot(epochs_range, history["val_acc"], "o-", label="validation")
axes[2].set_xlabel("epoch")
axes[2].set_ylabel("accuracy")
axes[2].set_title("Accuracy per epoch")
axes[2].legend()

plt.tight_layout()
plt.show()

best_epoch = int(np.argmax(history["val_acc"])) + 1
print(f"Best validation accuracy: {max(history['val_acc']):.2%} at epoch {best_epoch}")
print(f"Final train-validation accuracy gap: {history['train_acc'][-1] - history['val_acc'][-1]:.2%}")

## Cell 12 — Final exam: evaluate on the unseen TEST set

Now, and only now, we use the 10,000 test images: written by different people, never used for training or for choosing hyperparameters. **Test accuracy is the honest estimate of how the model will perform on new handwriting.** Training accuracy only tells you how well it memorized what it already saw.

We also print:
- **Confusion matrix:** row = true digit, column = predicted digit. The diagonal holds correct predictions; anything off-diagonal is a mistake (e.g. 4 ↔ 9, 3 ↔ 5, 7 ↔ 2 are typical confusions).
- **Classification report:** per-digit *precision* (when the model says "7", how often is it right?) and *recall* (of all real 7s, how many did it find?).

**Expected output:** ``TEST accuracy: 97.69%  (9769 / 10000 correct)` in our reference run` (anything ≥ 97% means everything works).

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

test_loss, test_acc = evaluate(model, X_test, y_test)
print(f"TEST loss:     {test_loss:.4f}")
print(f"TEST accuracy: {test_acc:.2%}  ({int(round(test_acc * len(y_test)))} / {len(y_test)} correct)")

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("predicted digit")
ax.set_ylabel("true digit")
ax.set_title("Confusion matrix on the test set")
plt.colorbar(im)
plt.show()

print(classification_report(y_test, y_pred, digits=4))

## Cell 13 — See predictions on unseen images (and the mistakes)

**Top grid:** 15 random test images. Green = correct, red = wrong. The percentage is the model's confidence (the softmax probability of its chosen digit).
**Bottom grid:** some test images the model got wrong. Many are genuinely ambiguous even for humans — a useful reminder that 100% is not a realistic goal.

In [ ]:
def show_predictions(model, images, labels, indices, title):
    probs = model.predict_proba(preprocess(images[indices]))
    preds = probs.argmax(axis=1)
    fig, axes = plt.subplots(1, len(indices), figsize=(1.6 * len(indices), 2.2))
    for ax, i, pred, p in zip(axes, indices, preds, probs):
        ax.imshow(images[i], cmap="gray")
        color = "green" if pred == labels[i] else "red"
        ax.set_title(f"pred {pred} ({p[pred]:.0%})\ntrue {labels[i]}", color=color, fontsize=9)
        ax.axis("off")
    plt.suptitle(title)
    plt.show()


rng_view = np.random.default_rng(7)
show_predictions(model, x_test, y_test, rng_view.choice(len(x_test), 15, replace=False),
                 "Random unseen test images")

wrong = np.where(y_pred != y_test)[0]
print(f"The model got {len(wrong)} of {len(y_test)} test images wrong.")
show_predictions(model, x_test, y_test, wrong[:15], "Some test images the model got wrong")

## Cell 14 — A prediction function for new inputs

`predict_digit()` is the "product" of Phase 1: give it any 28×28 image and it returns the digit and all 10 probabilities. Phase 4 (the app) will call exactly this kind of function.

**Input format the model expects** (same as MNIST): a white digit on a black background, roughly centered, values 0–255 (or already 0–1).

**Expected output:** a test image, the predicted digit with its confidence, and a bar chart of the 10 probabilities.

In [ ]:
def predict_digit(model, image):
    # image: array with 784 values (28x28), white digit on black background, 0-255 or 0-1
    x = np.asarray(image, dtype=np.float32).reshape(1, 784)
    if x.max() > 1.0:
        x = x / 255.0
    probs = model.predict_proba(x)[0]
    return int(np.argmax(probs)), probs


def show_digit_prediction(model, image, true_label=None):
    digit, probs = predict_digit(model, image)
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(8, 3))
    ax_img.imshow(np.asarray(image).reshape(28, 28), cmap="gray")
    ax_img.set_title(f"prediction: {digit} ({probs[digit]:.1%})"
                     + ("" if true_label is None else f" | true: {true_label}"))
    ax_img.axis("off")
    ax_bar.bar(range(10), probs)
    ax_bar.set_xticks(range(10))
    ax_bar.set_ylim(0, 1)
    ax_bar.set_xlabel("digit")
    ax_bar.set_ylabel("probability")
    plt.tight_layout()
    plt.show()
    return digit, probs


digit, probs = show_digit_prediction(model, x_test[0], true_label=y_test[0])
print("Predicted digit:", digit)
print("All probabilities:", np.round(probs, 4))

## Cell 15 (optional) — Try your own handwriting

1. Write a digit with a **thick dark marker on white paper**, take a photo, crop it to roughly a square around the digit.
2. Set `USE_MY_OWN_IMAGE = True` below, run the cell and upload the image.

The helper converts it to MNIST style: grayscale → invert colors (MNIST is white-on-black) → crop to the digit → resize to 20×20 → center in a 28×28 frame.

⚠️ **Expect lower accuracy than on the test set.** Your photo comes from a *different distribution* (lighting, pen, camera, centering) than MNIST. This is called **distribution shift**, and it's one of the main reasons real AI systems fail outside the lab. Good lesson to experience yourself!

With `USE_MY_OWN_IMAGE = False` the cell runs a self-test instead: it turns test image #1 (a "2") into a fake black-on-white photo and converts it back. In our reference run the model still answered **2**, but only with **66.5%** confidence (instead of ~100%), because our cropping and centering differ slightly from how MNIST was made. Even a tiny distribution shift is visible!

In [ ]:
from PIL import Image

USE_MY_OWN_IMAGE = False  # set to True to upload a photo (works in Google Colab)


def image_to_mnist_format(pil_image):
    gray = np.asarray(pil_image.convert("L"), dtype=np.float32)
    if gray.mean() > 127:                      # dark digit on light paper -> invert
        gray = 255.0 - gray
    gray[gray < 50] = 0                        # remove paper noise / shadows
    rows, cols = np.where(gray > 0)
    if len(rows) == 0:
        raise ValueError("No digit found in the image.")
    digit = gray[rows.min():rows.max() + 1, cols.min():cols.max() + 1]
    h, w = digit.shape
    scale = 20.0 / max(h, w)                   # MNIST digits fit in a 20x20 box
    new_size = (max(1, int(round(w * scale))), max(1, int(round(h * scale))))
    digit = np.asarray(Image.fromarray(digit).resize(new_size, Image.LANCZOS))
    canvas = np.zeros((28, 28), dtype=np.float32)
    top, left = (28 - digit.shape[0]) // 2, (28 - digit.shape[1]) // 2
    canvas[top:top + digit.shape[0], left:left + digit.shape[1]] = digit
    return np.clip(canvas, 0, 255)


if USE_MY_OWN_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded:
        mnist_like = image_to_mnist_format(Image.open(filename))
        show_digit_prediction(model, mnist_like)
else:
    # Self-test: turn a test image into a "photo" (black on white, larger) and convert it back
    fake_photo = Image.fromarray(255 - x_test[1]).resize((200, 200))
    converted = image_to_mnist_format(fake_photo)
    print("Conversion self-test, true label:", y_test[1])
    show_digit_prediction(model, converted, true_label=y_test[1])

## Cell 16 — Save, reload and download the trained model

The trained "intelligence" of Yassine AI is just these 101,770 numbers. We save them to a single `.npz` file (NumPy's compressed archive format) together with metadata (architecture, hyperparameters, test accuracy).

Then we **reload** the file into a brand-new network and check that it makes *exactly* the same predictions. Never trust a save function you haven't tested.

**Downloading:** in Colab, the file is downloaded to your computer automatically. You can also find it in the 📁 **Files** panel on the left (right-click → Download). ⚠️ Colab deletes files when the session ends, so download it (or save it to Google Drive — see the commented lines).

**Expected output:** `Saved model to yassine_ai_mnist_v1.npz (~400 KB)`, then `Reloaded model gives identical predictions: True`.

In [ ]:
MODEL_PATH = "yassine_ai_mnist_v1.npz"


def save_model(model, path, metadata):
    np.savez_compressed(path, metadata=np.array(json.dumps(metadata)), **model.params)


def load_model(path):
    with np.load(path) as data:
        metadata = json.loads(str(data["metadata"]))
        params = {name: data[name] for name in ["W1", "b1", "W2", "b2"]}
    input_size, hidden_size = params["W1"].shape
    output_size = params["W2"].shape[1]
    loaded = YassineNet(input_size, hidden_size, output_size)
    loaded.params = params
    return loaded, metadata


metadata = {
    "name": "Yassine AI - MNIST digit classifier v1",
    "architecture": f"784 -> {HIDDEN_SIZE} (ReLU) -> 10 (softmax)",
    "parameters": model.num_parameters(),
    "hyperparameters": {"hidden_size": HIDDEN_SIZE, "learning_rate": LEARNING_RATE,
                        "batch_size": BATCH_SIZE, "epochs": EPOCHS, "seed": SEED},
    "val_accuracy": history["val_acc"][-1],
    "test_accuracy": test_acc,
    "trained_on": "MNIST, 50,000 training images, random initialization (no pretrained weights)",
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
}
save_model(model, MODEL_PATH, metadata)
print(f"Saved model to {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024:.0f} KB)")

reloaded_model, reloaded_metadata = load_model(MODEL_PATH)
same_predictions = bool(np.array_equal(reloaded_model.predict(X_test), model.predict(X_test)))
print("Reloaded model gives identical predictions:", same_predictions)
print(json.dumps(reloaded_metadata, indent=2))

try:
    from google.colab import files
    files.download(MODEL_PATH)
except ImportError:
    print("Not running in Colab: the file is in the current folder:", os.path.abspath(MODEL_PATH))

# Optional: keep a permanent copy in your Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil; shutil.copy(MODEL_PATH, "/content/drive/MyDrive/" + MODEL_PATH)

## Cell 17 — Phase 1 success checklist (automatic)

Phase 1 is complete only when **every line is ✅**. If something is ❌, the message tells you where to look.

In [ ]:
checks = [
    ("Data loaded: 50k train / 10k val / 10k test",
     X_train.shape == (50000, 784) and X_val.shape == (10000, 784) and X_test.shape == (10000, 784),
     "Re-run Cells 2 and 4."),
    ("Backpropagation verified by gradient check", gradient_check_passed,
     "Bug in YassineNet.backward() (Cell 7)."),
    ("Training loss decreased", history["train_loss"][-1] < history["train_loss"][0],
     "Learning rate too small/large, or a bug in update()."),
    ("No NaN in training", bool(np.all(np.isfinite(history["batch_loss"]))),
     "Lower LEARNING_RATE (Cell 9)."),
    ("Validation accuracy >= 97%", history["val_acc"][-1] >= 0.97,
     "Train longer or check preprocessing (pixels must be divided by 255)."),
    ("Test accuracy >= 97% on unseen data", test_acc >= 0.97,
     "See validation item; never tune on the test set."),
    ("Model saved and reloads identically", same_predictions and os.path.exists(MODEL_PATH),
     "Check save_model()/load_model() (Cell 16)."),
]
for description, ok, hint in checks:
    print(("✅ " if ok else "❌ ") + description + ("" if ok else f"  ->  {hint}"))
print("\nPHASE 1 COMPLETE 🎉" if all(ok for _, ok, _ in checks) else "\nPhase 1 not complete yet.")

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `NameError: name 'X_train' is not defined` | A cell was skipped or the session restarted | **Runtime → Run all**, or run cells from the top |
| Loss is `nan` or jumps up and down | Learning rate too high | Set `LEARNING_RATE = 0.05` (or lower) in Cell 9, re-run Cells 9–17 |
| Accuracy stuck around 10% | Pixels not normalized, or labels shuffled differently from images | Check Cell 4: `/ 255.0`, and that `X` and `y` are split together |
| Initial loss far from 2.30 | Initialization scale wrong | Check `np.sqrt(2.0 / input_size)` in Cell 7 |
| Gradient check FAILED | Bug in `backward()` | Compare each line with the table in Cell 7's text |
| Download in Cell 2 fails | Network issue | The code falls back to OpenML automatically; otherwise re-run the cell |
| `files.download` does nothing | Browser blocked the download | Use the 📁 Files panel → right-click the `.npz` → Download |

## Experiments to deepen your understanding (do at least two)
1. `LEARNING_RATE = 1.0` → what happens to the loss? Then `0.001` → how fast does it learn?
2. `HIDDEN_SIZE = 16` vs. `512` → how do test accuracy and the train/val gap change?
3. Replace `relu` with no activation (identity) in `forward()` → the network becomes linear. What accuracy do you get? (Hint: ~92%, like simple logistic regression.)
4. Train on only 1,000 images (`X_train[:1000]`) for 50 epochs → watch **overfitting** appear in the Cell 11 plots.

## What's next
**Phase 2** turns this notebook into a clean, testable project: a Python package (`model.py`, `train.py`, `predict.py`), a README, reproducible training from the command line, plus a deeper look at overfitting and regularization.